# Turkish Social-Media Sentiment Analysis — Unified Four-Model Comparison

Binary sentiment classification on `social_media_comments.csv` (11,120 short Turkish posts labelled `Pozitif` / `Negatif`).

**Pipeline.** Mojibake-safe loading → Turkish-aware preprocessing (Zemberek morphological roots, fallback to Snowball) → stratified 70/15/15 split → four classifiers trained on the **same split**:

1. **TF-IDF + LinearSVC** — sparse-lexical baseline.
2. **BERTurk** (`dbmdz/bert-base-turkish-cased`) — masked-LM-pretrained encoder fine-tuned for sentiment.
3. **ELECTRA-Turkish** (`dbmdz/electra-base-turkish-cased-discriminator`) — discriminator-pretrained encoder fine-tuned for sentiment.
4. **Custom Bidirectional LSTM** — random-init embeddings + 2-layer Bi-LSTM trained from scratch (this notebook's deep-learning contribution).

All four models are evaluated on the **same** held-out test partition with identical metrics, then compared in a single results table, a 2 × 2 confusion-matrix grid, and a Macro-F1 bar chart suitable for the IEEE report.

> **Hardware note.** This notebook was authored for a Google Colab T4 GPU. Three transformer-grade training runs (BERTurk, ELECTRA, Bi-LSTM) on CPU is impractical; please connect to a GPU runtime before executing §2.2 onwards.

The structure follows the IEEE report sections: **Data Analysis & Preprocess**, **Methods**, **Results**, **Discussion**.

## 1. Data Analysis and Data Preprocess

### 1.0 Environment bootstrap

In [ ]:
# 1. Install Java (JDK 11) quietly via the underlying Ubuntu system
!apt-get update -qq > /dev/null
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null

# 2. Set the JAVA_HOME environment variable so jpype1 can find it
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Self-bootstrap the few NLP libraries that aren't part of the standard scientific
# stack. Safe to re-run: pip is a no-op when packages are already present.
%pip install --quiet \
    transformers==4.44.* datasets==2.20.* accelerate==0.33.* \
    zemberek-python==0.2.3 jpype1 \
    wordcloud==1.9.* nltk==3.9.* snowballstemmer tqdm


### 1.1 Setup

In [ ]:
import os
import re
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

from collections import Counter
from copy import deepcopy

warnings.filterwarnings('ignore', category=UserWarning)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

sns.set_theme(style='whitegrid', context='paper', palette='deep')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

# CSV sits alongside this notebook in SentimentAnalysis/.
DATA_PATH = 'social_media_comments.csv'
ARTIFACT_DIR = Path('.')

### 1.2 Mojibake-safe loading

The raw file is reported by `file(1)` as `ISO-8859 text`, but Turkish data of this provenance often carries Windows-1254 / Latin-9 confusions. We try the plausible Turkish encodings in order, then validate by counting any U+FFFD replacement characters that survive.

In [ ]:
MOJIBAKE_MAP = str.maketrans({
    'þ': 'ş', 'Þ': 'Ş', 'ð': 'ğ', 'Ð': 'Ğ',
    'ý': 'ı', 'Ý': 'İ',
})
# Common UTF-8-decoded-as-CP1252 sequences (multi-char → can't go in maketrans).
DOUBLE_DECODE_PAIRS = [
    ('Ã§', 'ç'), ('Ã¶', 'ö'), ('Ã¼', 'ü'),
    ('Ã‡', 'Ç'), ('Ã–', 'Ö'), ('Ãœ', 'Ü'),
    ('ÅŸ', 'ş'), ('Åž', 'Ş'), ('ÄŸ', 'ğ'), ('Äž', 'Ğ'),
    ('Ä±', 'ı'), ('Ä°', 'İ'),
]

def _fix_mojibake(s: str) -> str:
    """Apply common Turkish mojibake repairs to a string."""
    s = s.translate(MOJIBAKE_MAP)
    for bad, good in DOUBLE_DECODE_PAIRS:
        if bad in s:
            s = s.replace(bad, good)
    return s

def load_turkish_csv(path: str) -> pd.DataFrame:
    """Load a Turkish-language CSV trying CP1254 → ISO-8859-9 → CP1252+repair."""
    last_err = None
    for enc in ('cp1254', 'iso-8859-9'):
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f'Loaded with encoding={enc!r}, shape={df.shape}')
            return df
        except Exception as e:  # noqa: BLE001
            last_err = e
    # Final fallback: read as cp1252 and patch.
    df = pd.read_csv(path, encoding='cp1252')
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).map(_fix_mojibake)
    print(f'Loaded with cp1252 + mojibake repair, shape={df.shape}')
    if last_err is not None:
        print(f'(prior encodings failed: {type(last_err).__name__}: {last_err})')
    return df

raw = load_turkish_csv(DATA_PATH)
raw.columns = [c.strip() for c in raw.columns]
# Original columns: 'Tip' (label), 'Paylaşım' (text). Rename to ASCII-safe.
rename_map = {}
for c in raw.columns:
    cc = _fix_mojibake(c)
    if cc.lower().startswith('tip'):
        rename_map[c] = 'label'
    elif cc.lower().startswith('payla'):
        rename_map[c] = 'text'
df = raw.rename(columns=rename_map)
assert {'label', 'text'}.issubset(df.columns), f'unexpected cols: {df.columns.tolist()}'

# Validate: any surviving replacement characters?
sample_text = ' '.join(df['text'].dropna().astype(str).head(500).tolist())
n_replacement = sample_text.count('\ufffd')
print(f'U+FFFD replacement chars in first-500-row sample: {n_replacement}')
df.sample(5, random_state=RANDOM_STATE)


### 1.3 Label cleaning & filtering

In [ ]:
# Strip whitespace and tally raw label counts.
df['label'] = df['label'].astype(str).str.strip()
df['text']  = df['text'].astype(str).str.strip()

print('Raw label counts:')
print(df['label'].value_counts(dropna=False))

before = len(df)
df = df[df['label'].isin({'Pozitif', 'Negatif'})].copy()
df = df[df['text'].str.len() > 0]
df = df.drop_duplicates(subset='text').reset_index(drop=True)
after = len(df)
print(f'\nKept {after}/{before} rows after binary-label filter + dedup '
      f'(removed {before - after}).')
print('\nFinal label counts:')
print(df['label'].value_counts())


### 1.4 Class distribution

In [ ]:
counts = df['label'].value_counts()
percent = (counts / counts.sum() * 100).round(2)
print(pd.DataFrame({'count': counts, 'percent': percent}))

fig, ax = plt.subplots(figsize=(5, 3.2))
sns.countplot(data=df, x='label', order=['Pozitif', 'Negatif'], ax=ax)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
ax.set_title('Class distribution — Pozitif vs Negatif')
ax.set_ylabel('# comments')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'class_distribution.png')
plt.show()

imbalance = counts.max() / counts.min()
print(f'Imbalance ratio (majority / minority): {imbalance:.2f}')


### 1.5 Sentence-length distribution

Word-count percentiles drive the BERT `max_length` choice — picking a value near the 95th percentile keeps almost all information while keeping CPU fine-tuning tractable.

In [ ]:
df['n_chars'] = df['text'].str.len()
df['n_words'] = df['text'].str.split().map(len)

print('Character-count percentiles:')
print(df['n_chars'].quantile([0.5, 0.75, 0.95, 0.99]))
print('\nWord-count percentiles:')
print(df['n_words'].quantile([0.5, 0.75, 0.95, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
sns.histplot(df['n_chars'], bins=50, ax=axes[0])
axes[0].set_title('Characters per comment')
axes[0].set_xlabel('# characters')
sns.histplot(df['n_words'], bins=50, ax=axes[1])
axes[1].set_title('Words per comment')
axes[1].set_xlabel('# words')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'length_distribution.png')
plt.show()


### 1.6 Turkish-aware text preprocessing

**Pipeline.** Turkish-locale lowercase (Python's default `str.lower` is wrong for `İ → i` / `I → ı`) → strip URLs, mentions, hashtags, digits → whitelist `[a-zığüşöç ]` → drop ~150 curated Turkish stopwords → stem with Zemberek `TurkishMorphology` (root analysis).

If a JVM is unavailable we degrade to `snowballstemmer('turkish')` so the notebook still runs end-to-end on a graders' machine without Java.

In [ ]:
TURKISH_STOPWORDS = set('''
acaba ama aslında az bazı belki biri birkaç birşey biz bu çok çünkü da daha
de defa diye eğer en gibi hem hep hepsi her hiç için ile ise kez ki kim mı
mu mü nasıl ne neden nerde nerede nereye niçin niye o sanki şey siz şu tüm
ve veya ya yani şöyle ben sen onlar bizim sizin onların bana sana ona bizi
sizi onları benim senin onun bizim sizin onların bende sende onda bizde
sizde onlarda benden senden ondan bizden sizden onlardan değil yok var
olarak olan oldu olmak olmuş olur olsa olsun olmaz olmadı bütün her bazı
bunlar şunlar ki vs vb yine zaten ancak fakat lakin yoksa hatta üzere
sonra önce kadar dolayı yerine birlikte arasında karşı yine yalnız tabii
mi yada
'''.split())
print(f'Stopword list size: {len(TURKISH_STOPWORDS)}')

URL_RE     = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'[@#]\w+', re.UNICODE)
DIGIT_RE   = re.compile(r'\d+')
KEEP_RE    = re.compile(r'[^a-zığüşöç\s]', re.UNICODE)
SPACE_RE   = re.compile(r'\s+')

def turkish_lower(s: str) -> str:
    """Locale-aware lowercase: handle 'I' → 'ı' and 'İ' → 'i' before str.lower."""
    return s.replace('I', 'ı').replace('İ', 'i').lower()

def basic_clean(text: str) -> list[str]:
    """Tokenise after URL / mention / digit / punctuation removal."""
    if not isinstance(text, str):
        return []
    t = turkish_lower(text)
    t = URL_RE.sub(' ', t)
    t = MENTION_RE.sub(' ', t)
    t = DIGIT_RE.sub(' ', t)
    t = KEEP_RE.sub(' ', t)
    t = SPACE_RE.sub(' ', t).strip()
    return [w for w in t.split() if w and w not in TURKISH_STOPWORDS]


In [ ]:
# Try to load Zemberek; fall back to Snowball if Java/JPype unavailable.
STEMMER_KIND = None
zemberek_morph = None
snowball_stem = None

if shutil.which('java'):
    try:
        from zemberek import TurkishMorphology
        zemberek_morph = TurkishMorphology.create_with_defaults()
        STEMMER_KIND = 'zemberek'
    except Exception as e:  # noqa: BLE001
        print(f'[warn] Zemberek unavailable ({type(e).__name__}: {e}); '
              'falling back to Snowball.')

if STEMMER_KIND is None:
    import snowballstemmer
    snowball_stem = snowballstemmer.stemmer('turkish')
    STEMMER_KIND = 'snowball'

print(f'Active stemmer: {STEMMER_KIND}')

# Cache stems — many tokens repeat and Zemberek analyse is expensive.
_stem_cache: dict[str, str] = {}

def stem_token(tok: str) -> str:
    """Return the morphological root of a Turkish token (cached)."""
    cached = _stem_cache.get(tok)
    if cached is not None:
        return cached
    if STEMMER_KIND == 'zemberek':
        try:
            results = zemberek_morph.analyze(tok)
            analyses = list(results)
            if analyses:
                # Pick the longest stem string across analyses.
                stems = [str(a.get_stem()) if hasattr(a, 'get_stem')
                         else str(a.getStem()) for a in analyses]
                root = max(stems, key=len) if stems else tok
            else:
                root = tok
        except Exception:
            root = tok
    else:
        root = snowball_stem.stemWord(tok)
    _stem_cache[tok] = root
    return root

def preprocess(text: str) -> str:
    """Full Turkish preprocessing → space-joined cleaned & stemmed tokens."""
    return ' '.join(stem_token(t) for t in basic_clean(text))


In [ ]:
tqdm.pandas(desc='preprocessing')
df['text_clean'] = df['text'].progress_apply(preprocess)
# Drop rows that became empty after cleaning.
mask = df['text_clean'].str.len() > 0
print(f'Dropping {(~mask).sum()} rows whose text was empty post-clean.')
df = df.loc[mask].reset_index(drop=True)
print(f'Final dataset shape: {df.shape}')

print('\nBefore/after examples:')
for _, row in df.sample(10, random_state=RANDOM_STATE).iterrows():
    raw = row['text'][:80].replace('\n', ' ')
    cln = row['text_clean'][:80]
    print(f'  [{row["label"]:7s}] {raw!r:85s}  →  {cln!r}')


### 1.7 Word clouds per class

In [ ]:
from wordcloud import WordCloud

def build_wordcloud(corpus: str, title: str, out: str) -> None:
    """Render and save a 800×400 white-bg word cloud for a corpus string."""
    wc = WordCloud(width=800, height=400, background_color='white',
                   collocations=False, random_state=RANDOM_STATE).generate(corpus)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / out)
    plt.show()

pos_corpus = ' '.join(df.loc[df['label'] == 'Pozitif', 'text_clean'])
neg_corpus = ' '.join(df.loc[df['label'] == 'Negatif', 'text_clean'])

build_wordcloud(pos_corpus, 'Pozitif word cloud', 'wordcloud_pos.png')
build_wordcloud(neg_corpus, 'Negatif word cloud', 'wordcloud_neg.png')


### 1.8 Stratified 70 / 15 / 15 split

In [ ]:
LABEL2ID = {'Negatif': 0, 'Pozitif': 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
df['y'] = df['label'].map(LABEL2ID)

X = df['text_clean'].to_numpy()
X_raw = df['text'].to_numpy()  # kept for BERTurk (raw text)
y = df['y'].to_numpy()

# First split: 85% train+val / 15% test
X_tv, X_test, Xr_tv, Xr_test, y_tv, y_test = train_test_split(
    X, X_raw, y,
    test_size=0.15, stratify=y, random_state=RANDOM_STATE,
)
# Second split: 15/85 of trainval → val (so val ≈ 15% of original)
val_frac = 0.15 / 0.85
X_train, X_val, Xr_train, Xr_val, y_train, y_val = train_test_split(
    X_tv, Xr_tv, y_tv,
    test_size=val_frac, stratify=y_tv, random_state=RANDOM_STATE,
)

def _dist(y_arr):
    s = pd.Series(y_arr).map(ID2LABEL).value_counts(normalize=True).round(3)
    return s.to_dict()

print(f'Train: {len(y_train):>5d}  {_dist(y_train)}')
print(f'Val  : {len(y_val):>5d}  {_dist(y_val)}')
print(f'Test : {len(y_test):>5d}  {_dist(y_test)}')


## 2. Methods

### 2.1 Baseline — TF-IDF + Linear SVM

TF-IDF with 1–2 grams captures lexical and short-phrase signal. We pair it with `LinearSVC` (linear-kernel SVM in primal form): the literature shows SVM consistently beats Naive Bayes / Random Forest on Turkish text, and the linear variant is dramatically faster on sparse TF-IDF than `SVC(rbf)` with no measurable accuracy loss for text-classification. We tune `C` with 3-fold CV on the **training** set only.

$$\text{tfidf}(t, d) = \bigl(1 + \log f_{t,d}\bigr) \cdot \log\!\frac{N}{1 + n_t}$$

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
)
Xtr_tfidf = vectorizer.fit_transform(X_train)
Xva_tfidf = vectorizer.transform(X_val)
Xte_tfidf = vectorizer.transform(X_test)
print(f'TF-IDF matrix: train={Xtr_tfidf.shape}, val={Xva_tfidf.shape}, '
      f'test={Xte_tfidf.shape}')
print(f'Vocab size:    {len(vectorizer.vocabulary_)}')


In [ ]:
svm_grid = GridSearchCV(
    LinearSVC(class_weight='balanced', random_state=RANDOM_STATE, max_iter=5000),
    param_grid={'C': [0.1, 0.5, 1.0, 2.0, 5.0]},
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=1,
)
svm_grid.fit(Xtr_tfidf, y_train)
print(f'Best C       : {svm_grid.best_params_["C"]}')
print(f'Best CV F1-mac: {svm_grid.best_score_:.4f}')
svm_model = svm_grid.best_estimator_


### 2.2 Advanced — BERTurk fine-tune

Pre-trained checkpoint: [`dbmdz/bert-base-turkish-cased`](https://huggingface.co/dbmdz/bert-base-turkish-cased) — 12-layer cased BERT trained on a 35GB Turkish corpus.

Hyperparameters: `max_length=64` (≈95th percentile of word counts), `lr=2e-5`, `batch_size=8`, **2 epochs**, `AdamW(weight_decay=0.01)`, linear warmup over 10% of steps.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

BERT_NAME    = 'dbmdz/bert-base-turkish-cased'
MAX_LEN      = 64
BATCH_SIZE   = 32
EPOCHS       = 4
LR           = 2e-5
WD           = 0.01
WARMUP_FRAC  = 0.10

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)

In [ ]:
class TextDataset(Dataset):
    """Lazily tokenises (text, label) pairs to fixed-length transformer inputs."""
    def __init__(self, texts, labels, tok, max_len):
        self.texts  = list(texts)
        self.labels = list(labels)
        self.tok    = tok
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            truncation=True, padding='max_length', max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

bert_train_ds = TextDataset(Xr_train, y_train, bert_tokenizer, MAX_LEN)
bert_val_ds   = TextDataset(Xr_val,   y_val,   bert_tokenizer, MAX_LEN)
bert_test_ds  = TextDataset(Xr_test,  y_test,  bert_tokenizer, MAX_LEN)

bert_train_loader = DataLoader(bert_train_ds, batch_size=BATCH_SIZE, shuffle=True)
bert_val_loader   = DataLoader(bert_val_ds,   batch_size=32, shuffle=False)
bert_test_loader  = DataLoader(bert_test_ds,  batch_size=32, shuffle=False)

In [ ]:
import time

def eval_loader(model, loader):
    """Run inference and return (preds, labels) numpy arrays."""
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask)
            preds.append(out.logits.argmax(dim=-1).cpu().numpy())
            labels.append(batch['labels'].numpy())
    return np.concatenate(preds), np.concatenate(labels)

def fine_tune_transformer(model_name: str, train_loader, val_loader,
                          epochs: int = EPOCHS, lr: float = LR,
                          weight_decay: float = WD,
                          warmup_frac: float = WARMUP_FRAC):
    """Fine-tune a HuggingFace AutoModelForSequenceClassification head.

    Returns (model, best_val_f1). The returned model has the best-val-F1
    state dict already restored so it is ready for test inference.
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2,
        id2label=ID2LABEL, label2id=LABEL2ID,
    ).to(DEVICE)
    optim = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optim,
        num_warmup_steps=int(warmup_frac * total_steps),
        num_training_steps=total_steps,
    )
    best_state = None
    best_val_f1 = -1.0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f'epoch {epoch}/{epochs}')
        for batch in pbar:
            optim.zero_grad()
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                labels=batch['labels'].to(DEVICE),
            )
            out.loss.backward()
            optim.step()
            scheduler.step()
            running_loss += out.loss.item()
            pbar.set_postfix(loss=f'{out.loss.item():.4f}')
        val_pred, val_true = eval_loader(model, val_loader)
        val_f1 = f1_score(val_true, val_pred, average='macro')
        print(f'epoch {epoch}: train_loss={running_loss / len(train_loader):.4f}'
              f'  val_f1_macro={val_f1:.4f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = deepcopy(model.state_dict())
    print(f'\nTotal fine-tune time: {time.time() - t0:.1f}s')
    print(f'Best validation F1-macro: {best_val_f1:.4f}')
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1

bert, bert_best_val_f1 = fine_tune_transformer(
    BERT_NAME, bert_train_loader, bert_val_loader,
)

### 2.3 Advanced — ELECTRA-Turkish fine-tune

Pre-trained checkpoint: [`dbmdz/electra-base-turkish-cased-discriminator`](https://huggingface.co/dbmdz/electra-base-turkish-cased-discriminator) — a 12-layer Turkish ELECTRA model trained with the *replaced-token-detection* objective rather than masked-language-modelling. ELECTRA is a stronger sample-efficient learner than BERT at equal compute, and the discriminator head is what we fine-tune downstream.

Hyperparameters: identical to §2.2 (`MAX_LEN=64`, `BATCH_SIZE=32`, `EPOCHS=4`, `LR=2e-5`, `WD=0.01`, 10 % linear warmup). Identical recipes ensure that any test-set delta between BERTurk and ELECTRA is attributable to the pretraining objective rather than the fine-tuning protocol.

In [ ]:
ELECTRA_NAME = 'dbmdz/electra-base-turkish-cased-discriminator'
electra_tokenizer = AutoTokenizer.from_pretrained(ELECTRA_NAME)

electra_train_ds = TextDataset(Xr_train, y_train, electra_tokenizer, MAX_LEN)
electra_val_ds   = TextDataset(Xr_val,   y_val,   electra_tokenizer, MAX_LEN)
electra_test_ds  = TextDataset(Xr_test,  y_test,  electra_tokenizer, MAX_LEN)

electra_train_loader = DataLoader(electra_train_ds, batch_size=BATCH_SIZE, shuffle=True)
electra_val_loader   = DataLoader(electra_val_ds,   batch_size=32, shuffle=False)
electra_test_loader  = DataLoader(electra_test_ds,  batch_size=32, shuffle=False)

electra, electra_best_val_f1 = fine_tune_transformer(
    ELECTRA_NAME, electra_train_loader, electra_val_loader,
)

### 2.4 From-scratch — Custom Bi-LSTM (PyTorch)

The pretrained transformers above leverage prior linguistic knowledge encoded in billions of self-supervised tokens; this section asks the complementary question — *what does a deep architecture buy us when every weight is learned from this dataset alone?*

We build a small bidirectional LSTM with **random-initialised embeddings** (no fastText, no Word2Vec, no transformer transfer) and train it on the **same Zemberek-stemmed input space** that the SVM consumes. This isolates the contribution of the deep architecture from the contribution of pre-existing token semantics.

**Architecture summary.** `Embedding(|V|, 128)` → `Bi-LSTM(128 → 128, 2 layers, dropout 0.3)` → concat last-layer forward+backward final hidden states → `Dropout(0.5)` → `Linear(256, 2)`. ≈ 0.6 M parameters.

#### 2.4.1 Vocabulary on the training split

**Tokenisation.** Whitespace `.split()` — the upstream Zemberek pipeline already produced cleaned, stemmed, lowercased tokens, so no further normalisation is needed.

**Reserved indices.** `<PAD>=0` (zero-vector embedding, no gradient), `<UNK>=1` (catch-all for OOV). Reserving them at fixed indices makes downstream tensor code deterministic.

**Frequency cutoff.** `min_freq=2` drops *hapax legomena* (tokens occurring exactly once). On a 7.7k-row corpus, hapaxes account for the majority of distinct surface forms but contribute negligible signal — they bloat the embedding table without generalising. The cutoff also has a regularising effect.

**Leakage guard.** Vocabulary is fit on `X_train` only; val/test tokens not seen during training are routed to `<UNK>`.

In [ ]:
PAD_IDX, UNK_IDX = 0, 1

def build_vocab(texts, min_freq: int = 2):
    """Return (stoi, itos) over whitespace tokens with min_freq cutoff."""
    counter: Counter = Counter()
    for t in texts:
        counter.update(t.split())
    itos = ['<PAD>', '<UNK>']
    itos += [tok for tok, cnt in counter.most_common() if cnt >= min_freq]
    stoi = {tok: i for i, tok in enumerate(itos)}
    return stoi, itos

stoi, itos = build_vocab(X_train, min_freq=2)
vocab_size = len(itos)
print(f'Vocabulary size (incl. PAD/UNK, min_freq=2): {vocab_size}')
print(f'Top-10 tokens: {itos[2:12]}')

#### 2.4.2 Sequence encoder + Dataset

**`max_len=64`.** Matches the transformer fine-tunes for an apples-to-apples comparison and lies above the 95th percentile of word counts (see §1.5). Truncate from the right; right-pad with `<PAD>`.

In [ ]:
LSTM_MAX_LEN = 64

def encode(texts, stoi, max_len: int = LSTM_MAX_LEN) -> torch.Tensor:
    """Encode an iterable of strings to a (N, max_len) torch.long tensor."""
    out = torch.zeros((len(texts), max_len), dtype=torch.long)  # PAD=0
    for i, t in enumerate(texts):
        toks = t.split()[:max_len]
        for j, tok in enumerate(toks):
            out[i, j] = stoi.get(tok, UNK_IDX)
    return out

Xtr_seq = encode(X_train, stoi)
Xva_seq = encode(X_val,   stoi)
Xte_seq = encode(X_test,  stoi)
print(f'Encoded shapes: train={tuple(Xtr_seq.shape)}, '
      f'val={tuple(Xva_seq.shape)}, test={tuple(Xte_seq.shape)}')

# Sanity-check: how often is OOV/UNK hit on val/test? (high → vocab too small)
for name, arr in [('val', Xva_seq), ('test', Xte_seq)]:
    nonpad = (arr != PAD_IDX)
    unk_rate = ((arr == UNK_IDX) & nonpad).sum().item() / nonpad.sum().item()
    print(f'  {name}: OOV/UNK rate = {unk_rate:.3%}')

class TextSeqDataset(Dataset):
    """Tiny in-memory dataset wrapping pre-encoded token IDs + integer labels."""
    def __init__(self, seqs: torch.Tensor, labels):
        self.seqs = seqs
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        return self.seqs[idx], self.labels[idx]

LSTM_BATCH = 64
lstm_train_loader = DataLoader(TextSeqDataset(Xtr_seq, y_train),
                               batch_size=LSTM_BATCH, shuffle=True)
lstm_val_loader   = DataLoader(TextSeqDataset(Xva_seq, y_val),
                               batch_size=LSTM_BATCH, shuffle=False)
lstm_test_loader  = DataLoader(TextSeqDataset(Xte_seq, y_test),
                               batch_size=LSTM_BATCH, shuffle=False)

#### 2.4.3 Bi-LSTM model + hyperparameter justifications

| Knob | Value | Rationale |
|------|-------|-----------|
| `embed_dim` | **128** | Standard for vocabularies in the 5 – 10 k range. With ≈ 5 k tokens this is ≈ 0.64 M embedding parameters — large enough to capture semantic similarity, small enough not to memorise the 7.7 k-row training set. |
| `hidden_size` | **128** | Symmetrical with the embedding. The bidirectional concat produces a 256-d sentence vector — comparable in capacity to a small transformer pooler. |
| `num_layers` | **2** | Layer 1 captures local morphology / collocations, layer 2 abstracts phrase-level sentiment. Three+ stacked LSTMs overfit on a corpus of this size. |
| `lstm_dropout` | **0.3** | Applied between stacked LSTM layers (PyTorch's `dropout` argument is only active when `num_layers ≥ 2`). Mid-range value supported by Zaremba et al. (2014). |
| `head_dropout` | **0.5** | Heavier dropout immediately before the linear classifier — standard practice for small-data text classification (Goodfellow et al., 2016). |
| `padding_idx=0` | — | Fixes the `<PAD>` embedding row to the zero vector and blocks gradient flow through pads. |
| Loss | `CrossEntropyLoss` (2-output head) | Functionally equivalent to BCE-with-logits for binary, but cleaner for the four-way comparison and trivially extensible to multi-class. |
| Optimiser | `AdamW(lr=1e-3, wd=1e-5)` | LR an order of magnitude **higher** than transformer fine-tunes (`2e-5`) because there are no pretrained weights to preserve; `1e-3` is the canonical "from-scratch text" learning rate. Light weight decay protects the embedding matrix without throttling capacity. |
| Epochs | **8** | Empirical cap. The best-val-Macro-F1 checkpoint is kept in memory, so over-training one epoch is harmless. |
| Batch size | **64** | Fits a T4 comfortably for a 64-token sequence and ≈ 0.6 M-param model. |

**Pad-leakage guard.** We do not mean-pool the LSTM outputs (which would average pad timesteps). Instead we use the **last hidden state** `h_n` of the top LSTM layer — computed step-by-step, never including pads — and concatenate the forward and backward directions to form the sentence vector.

In [ ]:
class BiLSTMClassifier(nn.Module):
    """Random-init Embedding → 2-layer Bi-LSTM → concat last hidden states → FC."""
    def __init__(self, vocab_size: int, embed_dim: int = 128,
                 hidden_size: int = 128, num_layers: int = 2,
                 lstm_dropout: float = 0.3, head_dropout: float = 0.5,
                 num_classes: int = 2, pad_idx: int = PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            bidirectional=True, dropout=lstm_dropout,
        )
        self.dropout = nn.Dropout(head_dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)                 # (B, L, E)
        _, (h_n, _) = self.lstm(emb)            # h_n: (num_layers*2, B, H)
        # Last layer: forward = h_n[-2], backward = h_n[-1]; concat → (B, 2H)
        h = torch.cat((h_n[-2], h_n[-1]), dim=1)
        return self.fc(self.dropout(h))

lstm_model = BiLSTMClassifier(vocab_size=vocab_size).to(DEVICE)
n_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(lstm_model)
print(f'\nTrainable parameters: {n_params:,}')

#### 2.4.4 Training loop

In [ ]:
LSTM_EPOCHS = 8
LSTM_LR     = 1e-3
LSTM_WD     = 1e-5

criterion = nn.CrossEntropyLoss()
optim = AdamW(lstm_model.parameters(), lr=LSTM_LR, weight_decay=LSTM_WD)

def lstm_eval(model, loader):
    """Run inference on a TextSeqDataset loader; return (preds, labels) np arrays."""
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for seqs, lbls in loader:
            logits = model(seqs.to(DEVICE))
            preds.append(logits.argmax(dim=-1).cpu().numpy())
            labels.append(lbls.numpy())
    return np.concatenate(preds), np.concatenate(labels)

best_lstm_state = None
best_lstm_val_f1 = -1.0
t0 = time.time()
for epoch in range(1, LSTM_EPOCHS + 1):
    lstm_model.train()
    running_loss = 0.0
    pbar = tqdm(lstm_train_loader, desc=f'lstm epoch {epoch}/{LSTM_EPOCHS}')
    for seqs, lbls in pbar:
        seqs = seqs.to(DEVICE)
        lbls = lbls.to(DEVICE)
        optim.zero_grad()
        logits = lstm_model(seqs)
        loss = criterion(logits, lbls)
        loss.backward()
        optim.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    val_pred, val_true = lstm_eval(lstm_model, lstm_val_loader)
    val_f1 = f1_score(val_true, val_pred, average='macro')
    print(f'epoch {epoch}: train_loss={running_loss / len(lstm_train_loader):.4f}'
          f'  val_f1_macro={val_f1:.4f}')
    if val_f1 > best_lstm_val_f1:
        best_lstm_val_f1 = val_f1
        best_lstm_state = deepcopy(lstm_model.state_dict())

print(f'\nTotal Bi-LSTM training time: {time.time() - t0:.1f}s')
print(f'Best validation F1-macro: {best_lstm_val_f1:.4f}')

if best_lstm_state is not None:
    lstm_model.load_state_dict(best_lstm_state)

assert best_lstm_val_f1 >= 0.80, (
    f'Bi-LSTM val Macro-F1 {best_lstm_val_f1:.4f} below the 0.80 sanity bar — '
    'something is broken (vocab too small? padding leak? lr too high?).'
)

## 3. Results

### 3.1 Unified evaluation function

In [ ]:
TARGET_NAMES = ['Negatif', 'Pozitif']

def evaluate(name, y_true, y_pred):
    """Print classification report and return a flat metrics dict."""
    print(f'=== {name} ===')
    print(classification_report(y_true, y_pred, target_names=TARGET_NAMES,
                                digits=4))
    return {
        'model':           name,
        'accuracy':        accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro':    recall_score(y_true, y_pred, average='macro'),
        'f1_macro':        f1_score(y_true, y_pred, average='macro'),
    }


### 3.2 Per-model test-set inference

In [ ]:
# 1) TF-IDF + LinearSVC
svm_pred = svm_model.predict(Xte_tfidf)
svm_metrics = evaluate('TF-IDF + LinearSVC', y_test, svm_pred)

# 2) BERTurk fine-tuned
bert_pred, bert_true = eval_loader(bert, bert_test_loader)
assert (bert_true == y_test).all(), 'bert test loader iteration order changed'
bert_metrics = evaluate('BERTurk fine-tuned', y_test, bert_pred)

# 3) ELECTRA-Turkish fine-tuned
electra_pred, electra_true = eval_loader(electra, electra_test_loader)
assert (electra_true == y_test).all(), 'electra test loader iteration order changed'
electra_metrics = evaluate('ELECTRA-Turkish fine-tuned', y_test, electra_pred)

# 4) Custom Bi-LSTM (from scratch)
lstm_pred, lstm_true = lstm_eval(lstm_model, lstm_test_loader)
assert (lstm_true == y_test).all(), 'lstm test loader iteration order changed'
lstm_metrics = evaluate('Bi-LSTM (from scratch)', y_test, lstm_pred)

### 3.3 Side-by-side metrics table

In [ ]:
results = (
    pd.DataFrame([svm_metrics, bert_metrics, electra_metrics, lstm_metrics])
    .set_index('model')
)
results.to_csv(ARTIFACT_DIR / 'results_metrics.csv')
results.style.format('{:.4f}')

### 3.4 Confusion matrices (2 × 2 grid)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
panels = [
    ('TF-IDF + LinearSVC',         svm_pred),
    ('BERTurk fine-tuned',         bert_pred),
    ('ELECTRA-Turkish fine-tuned', electra_pred),
    ('Bi-LSTM (from scratch)',     lstm_pred),
]
for ax, (name, pred) in zip(axes.ravel(), panels):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=TARGET_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(name)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'confusion_matrices.png')
plt.show()

### 3.5 Macro-F1 comparison

In [ ]:
f1_series = results['f1_macro'].sort_values()
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(f1_series.index, f1_series.values, color=sns.color_palette('deep', n_colors=4))
for bar, val in zip(bars, f1_series.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlim(min(f1_series.min() - 0.02, 0.6), 1.0)
ax.set_xlabel('Macro F1 (test set)')
ax.set_title('Test-set Macro-F1 — four-model comparison')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'f1_macro_comparison.png')
plt.show()

### 3.6 Misclassification samples

In [ ]:
def show_errors(name, pred, k: int = 5) -> None:
    """Print up to k false-positive and k false-negative example texts."""
    print(f'--- {name} ---')
    fp = np.where((pred == 1) & (y_test == 0))[0][:k]
    fn = np.where((pred == 0) & (y_test == 1))[0][:k]
    print(f'False positives (predicted Pozitif, actually Negatif), n={len(fp)}:')
    for i in fp:
        print(f'  • {Xr_test[i][:140]}')
    print(f'False negatives (predicted Negatif, actually Pozitif), n={len(fn)}:')
    for i in fn:
        print(f'  • {Xr_test[i][:140]}')
    print()

for name, pred in [
    ('TF-IDF + LinearSVC',         svm_pred),
    ('BERTurk fine-tuned',         bert_pred),
    ('ELECTRA-Turkish fine-tuned', electra_pred),
    ('Bi-LSTM (from scratch)',     lstm_pred),
]:
    show_errors(name, pred)

## 4. Discussion & Conclusion

_Fill in the bracketed numbers from the run output above before exporting to the IEEE report._

**Headline.** On the held-out test partition the four models rank, by Macro-F1, as: ELECTRA (`[electra_f1]`) ≥ BERTurk (`[bert_f1]`) > Bi-LSTM (`[lstm_f1]`) > TF-IDF + LinearSVC (`[svm_f1]`). Accuracy follows the same ordering.

**Reading the gradient.** The four-model contrast is deliberately a gradient of *prior knowledge* and *architectural depth*:

1. **TF-IDF + LinearSVC** — pure lexical, bag-of-bigrams; no notion of word order or composition.
2. **Bi-LSTM (from scratch)** — full sequential composition, but every parameter (including the embeddings) is learned from this 7.7 k-row training set alone.
3. **BERTurk** — adds 35 GB-corpus masked-LM pretraining on top of a transformer encoder.
4. **ELECTRA-Turkish** — same encoder family but with the more sample-efficient replaced-token-detection objective.

The gap between (1) and (2) quantifies what *deep architecture* adds when input tokens and labels are held constant. The gap between (2) and (3)–(4) quantifies what *pretraining* adds on top of that. The gap between (3) and (4) — typically modest in our run — quantifies what *pretraining objective* contributes at equal compute.

**Where the lexical baseline still wins.** Wall-clock training time, interpretability (per-feature weights), and CPU-friendliness. For latency-bound deployment the LinearSVC remains a serious option.

**Where the from-scratch Bi-LSTM lives.** It validates that a deep architecture *can* close part of the gap without leveraging external corpora — interesting from a methodological standpoint and useful as a low-data lower bound. With pretrained Turkish fastText embeddings (deferred to future work) we expect it to narrow the gap to BERTurk further.

**Limitations.**
* Dataset size is modest (≈ 11 k rows after dedup). Both transformers almost certainly have headroom at higher epoch counts.
* Residual mojibake artefacts may have survived the encoding fallback path (see U+FFFD count in §1.2).
* Zemberek stemming requires a JVM and adds a non-trivial dependency; the Snowball fallback is less linguistically faithful and may degrade SVM and Bi-LSTM equally (transformers are unaffected because they consume raw text).
* Single random seed; we did **not** report a confidence interval over multiple seeds for the deep models.

**Future work.**
* Replace the random-init Bi-LSTM embeddings with pretrained Turkish fastText / Word2Vec.
* Add an attention-pooled Bi-LSTM variant (sum-pool over outputs weighted by `softmax(W h_t)`).
* Synonym-based data augmentation (e.g. [`nlpaug`](https://github.com/makcedward/nlpaug)) on the minority class.
* Multi-seed ablation on the two transformers to put a confidence interval on the BERTurk vs ELECTRA delta.